# FDS 통합 스코어링 파이프라인 — 4조 이다영**로그인 → 계정 변경 → 결제**를 하나의 계정 키로 이어 위험도를 산출하고,팀원이 만든 화면(`FDS UXUI prototype files/`)에 넣을 데이터를 생성한다.| 단계 | 기능 | 따르는 수업 || :--- | :--- | :--- || 1 | ① 노출 정황 스코어링 | **41회차** `gate × 가중합` || 2 | ② 계정 장악 감지 | **41회차** + **52회차** 누수 규율 || 3 | ③ 환금성 결제 판정 | **52회차** IsolationForest + **47회차** SHAP || 4 | 통합 점수 → `action` / `reason` | 41회차 || 5 | 운영 검토 큐 CSV | **42회차** 대시보드 입력 규격 |> ⚠️ 로그인·계정변경 데이터는 **합성**이다(`DATA_CARD.md`). 결제 데이터만 수업 제공분이다.

## 0. 라이브러리 · 데이터 로드

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
import json, warnings
warnings.filterwarnings("ignore")

def find(name):
    for b in [Path("."), Path("code"), Path("/content"),
              Path("/Users/leedayoung/fds-team-project/code")]:
        if (b/name).exists(): return b/name
    raise FileNotFoundError(name)

TX = pd.read_csv(find("transactions.csv"), encoding="utf-8-sig", parse_dates=["transaction_time"])
LG = pd.read_csv(find("login_events.csv"), encoding="utf-8-sig", parse_dates=["login_time"])
CH = pd.read_csv(find("account_change_events.csv"), encoding="utf-8-sig", parse_dates=["change_time"])

# 52회차 데이터 계약 검증 — 필수 컬럼이 없으면 여기서 멈춘다
REQUIRED = {
    "transactions": {"transaction_id","user_id","transaction_time","amount",
                     "merchant_category","channel","is_new_device","is_foreign_ip",
                     "retry_count","merchant_risk","is_fraud","location"},
    "login":        {"login_id","user_id","login_time","ip_addr","device_id",
                     "country","login_success","is_account_takeover"},
    "change":       {"change_id","user_id","change_time","change_type","device_id","country"},
}
for name, df, req in [("transactions",TX,REQUIRED["transactions"]),
                      ("login",LG,REQUIRED["login"]), ("change",CH,REQUIRED["change"])]:
    missing = sorted(req - set(df.columns))
    if missing: raise ValueError(f"[{name}] 필수 컬럼 없음: {missing}")
print("데이터 계약 통과")

TX = TX.sort_values("transaction_time").reset_index(drop=True)
LG = LG.sort_values("login_time").reset_index(drop=True)
print(f"결제 {len(TX):,} (사기 {int(TX.is_fraud.sum())}) | 로그인 {len(LG):,} (ATO {int(LG.is_account_takeover.sum())}) | 변경 {len(CH)}")
print(f"사용자 {TX.user_id.nunique()}명 | 기간 {TX.transaction_time.min().date()} ~ {TX.transaction_time.max().date()}")

---## 1. 누수 없는 파생 피처 (52회차 Part1 규율)52회차에서 배운 원칙 — **판정 시점에 알 수 있는 값만 쓴다.**`groupby().transform()`은 전체 기간을 보므로 미래 정보가 샌다.

In [ ]:
# ── ip_user_count : 그 시점까지 그 IP가 두드린 누적 고유 계정 수
# 🔴 누수: LG.groupby("ip_addr")["user_id"].transform("nunique")   ← 전체 기간을 봄
first_pair = (~LG.duplicated(subset=["ip_addr","user_id"])).astype(int)
LG["ip_user_count"] = first_pair.groupby(LG["ip_addr"]).cumsum()

# ── 직전 로그인 실패 횟수 (52회차 shift().expanding() 과 같은 원리)
LG["fail_run"] = (LG.groupby("user_id")["login_success"]
                    .transform(lambda s: (1-s).shift().rolling(5, min_periods=1).sum())
                    .fillna(0))

# ── 계정별 과거 평균 금액 (52회차 co15 '안전 버전' 그대로)
TX["past_mean_amount"] = (TX.groupby("user_id")["amount"]
                            .transform(lambda s: s.shift().expanding().mean()))
TX["amount_ratio"] = TX["amount"] / TX["past_mean_amount"]

print("[누수 점검] 전체기간 방식 vs 시점누적 방식")
leaky = LG.groupby("ip_addr")["user_id"].transform("nunique")
atk = LG.is_attack_ip == 1
print(f"  공격 IP 기준 평균  누수 {leaky[atk].mean():.1f}  →  보정 {LG.ip_user_count[atk].mean():.1f}")
print(f"  최댓값             누수 {leaky.max():.0f}  →  보정 {LG.ip_user_count.max():.0f}")
print(f"  첫 거래라 과거 평균이 없는 건: {int(TX.past_mean_amount.isna().sum())} / {len(TX)}")

---## 2. 기능① 노출 정황 스코어링 (41회차 `gate × 가중합`)```score = gate × (60·자사유출 + 40·다크웹 + 15·스터핑IP)````[데이터한계]` E1(자사 유출 명단)·E2(다크웹)는 실데이터가 없다.**E3(스터핑 IP 관측)만 실제로 계산**하고, E1·E2는 시뮬레이션으로 일부 계정에 부여한다.

In [ ]:
rng = np.random.default_rng(41)
users = sorted(TX.user_id.unique())
exp = pd.DataFrame({"user_id": users})

# E3 — 실제 계산: 이 계정을 두드린 IP 중 '가장 많은 계정을 건드린 IP'가 상위 10%인가
#
# ⚠️ 절대 임계값(ip_user_count >= 5)을 쓰면 398명 중 351명(88%)이 걸린다.
#    회사·카페 공용 IP가 정상적으로 20계정 넘게 쓰기 때문이다(DATA_CARD 참조).
#    선별이 목적인 점수에서 88%가 걸리면 아무것도 선별하지 못한다.
# → 41회차 minmax 정신대로 **상대 기준(분위수)**으로 바꾼다.
user_max_ipc = LG.groupby("user_id")["ip_user_count"].max()
STUFF_TH = float(user_max_ipc.quantile(0.90))
e3_users = set(user_max_ipc[user_max_ipc >= STUFF_TH].index)
exp["E3_stuffing_ip"] = exp.user_id.isin(e3_users).astype(int)
print(f"E3 임계값(상위 10% 분위수): ip_user_count >= {STUFF_TH:.0f}")

# E1·E2 — [시뮬레이션] 실데이터 없음. 발표에서 반드시 명시할 것
exp["E1_own_breach"] = (rng.random(len(exp)) < 0.03).astype(int)   # 3%
exp["E2_darkweb"]    = (rng.random(len(exp)) < 0.12).astype(int)   # 12%

# gate — 조치(비밀번호 변경·MFA)를 완료한 계정은 강화 대상에서 제외
changed_pw = set(CH.loc[CH.change_type == "password", "user_id"])
exp["gate"] = (~exp.user_id.isin(changed_pw)).astype(int)

exp["exposure_score"] = (exp.gate * (60*exp.E1_own_breach + 40*exp.E2_darkweb
                                     + 15*exp.E3_stuffing_ip)).clip(upper=100)

def exposure_action(s):
    if s >= 60: return "최대 강화"
    if s >= 40: return "강화"
    if s >= 15: return "관찰"
    return "평시"
exp["exposure_action"] = exp.exposure_score.apply(exposure_action)

print(f"E3(실측) 해당 계정: {int(exp.E3_stuffing_ip.sum())}명 / {len(exp)}명")
print(f"gate=0 (조치 완료로 제외): {int((exp.gate==0).sum())}명\n")
print(exp.exposure_action.value_counts().reindex(["평시","관찰","강화","최대 강화"]).to_string())

---## 3. 기능② 계정 장악 감지로그인 성공 후 **60분 내** 민감정보 변경을 트리거로 본다(T=60분은 실측으로 결정).### 팀원 화면에 맞춰 신호 하나를 추가했다 — **물리적 이동 불가**`shap-starbucks.html`에 *"몇 시간 전 국내에서 정상 결제가 있어요 · 물리적으로 이동 불가"*가 있다.이건 우리 명세에 없던 신호인데 **데이터로 계산 가능**하고 신호도 강해서 채택한다.

In [ ]:
T_WINDOW = 60   # 분 — 실측: 60분에서 재현율 포화

suc = LG[LG.login_success == 1][["user_id","login_time","device_id","country",
                                 "ip_user_count","fail_run","is_account_takeover"]]
suc.columns = ["user_id","login_ts","login_device","login_country",
               "ipc","fail_run","is_ato"]

ev = CH.merge(suc, on="user_id")
ev = ev[(ev.change_time > ev.login_ts) &
        (ev.change_time <= ev.login_ts + pd.Timedelta(minutes=T_WINDOW))]
ev = ev.sort_values("login_ts").groupby("change_id").tail(1).copy()   # 변경 직전 로그인

# ── 신호 6종
ev["s_new_device"]  = (~ev.login_device.str.match(r"DEV-\d+-[01]$")).astype(int)
ev["s_foreign"]     = (ev.login_country == "해외").astype(int)
ev["s_contact_chg"] = ev.change_type.isin(["email","phone"]).astype(int)
# ⚠️ 기능①의 STUFF_TH를 그대로 쓰면 안 된다. 두 기능은 **척도가 다르다.**
#    기능① : 계정 단위 — 5개월 동안 그 계정을 두드린 IP의 **최댓값** (상위 10% = 53)
#    기능② : 이벤트 단위 — **그 로그인 시점**의 ip_user_count (중앙값 1)
#    같은 임계값을 쓰면 기능②에서 해당 건수가 0이 된다.
LOGIN_MULTI_IP_TH = 5
ev["s_multi_ip"]    = (ev.ipc >= LOGIN_MULTI_IP_TH).astype(int)
ev["s_fail_run"]    = (ev.fail_run >= 1).astype(int)

# ★ 물리적 이동 불가 — 검토했으나 **이 데이터에서는 채택 불가**
#
#   팀원 화면(shap-starbucks.html)에 "몇 시간 전 국내 정상 결제가 있어요"가 있어 계산을 시도했다.
#   결과: 결제 기준 6~24시간 창에서 해당 0건, 로그인 기준으로 바꿔도 ATO 25건 중 1~2건뿐이었다.
#   원인은 결제 빈도가 사용자당 약 25일에 1건으로 너무 낮다는 것이다.
#   → 신호 자체는 타당하나 **이 데이터의 밀도로는 성립하지 않는다.** 배점을 주지 않고 제외한다.
#     (실서비스에서는 로그인이 훨씬 조밀하므로 유효한 신호다 — 오픈 이슈로 남긴다)

base = ev.is_ato.mean()
print(f"트리거된 이벤트 {len(ev)}건 | 그중 ATO {int(ev.is_ato.sum())}건 (기저 {base:.3f})\n")
print("[신호별 리프트 — 평활 전/후]")
A = 15   # Laplace 유사표본
SIGNALS = [("s_multi_ip","다계정 IP"),("s_foreign","낯선 국가"),
           ("s_contact_chg","연락처 변경"),("s_new_device","신규 기기"),("s_fail_run","직전 로그인 실패")]
lift = {}
for c,label in SIGNALS:
    s = ev[ev[c]==1]; n=len(s); tp=int(s.is_ato.sum())
    raw = (tp/n)/base if n else 0
    sm  = ((tp + A*base)/(n + A))/base
    lift[c] = sm
    print(f"  {label:14s} n={n:>3} | 원시 {raw:>4.2f} → 평활 {sm:>4.2f}")

### 배점은 평활 리프트에 비례해 정한다원시 리프트를 그대로 쓰면 **표본이 작은 신호에 과적합**된다.`다계정 IP`는 리프트가 가장 높지만 해당 건수가 한 자릿수라, Laplace 평활로 기저율 쪽으로 당긴다.**"신호가 세다"와 "그렇게 믿을 만큼 봤다"는 다르다.**

In [ ]:
w = np.array([lift[c] for c,_ in SIGNALS])
w = np.round(w / w.sum() * 100 / 5) * 5
w[0] += 100 - w.sum()                     # 합계 100 보정
WEIGHTS = {c: int(x) for (c,_), x in zip(SIGNALS, w)}

print("[기능② 배점]")
for c,label in SIGNALS: print(f"  {label:14s} {WEIGHTS[c]:>3}점")
print(f"  {'합계':14s} {sum(WEIGHTS.values()):>3}점")

ev["takeover_score"] = sum(WEIGHTS[c] * ev[c] for c,_ in SIGNALS)

def takeover_action(s):
    if s >= 80: return "계정 잠금"
    if s >= 60: return "변경 보류 + 원래 연락처 통보"
    if s >= 40: return "변경 반영 + 원래 연락처 통보"
    return "정상 처리"
ev["takeover_action"] = ev.takeover_score.apply(takeover_action)

def roc_auc(score, y):
    y = np.asarray(y); r = pd.Series(score).rank().values
    p, n = y==1, y==0
    return (r[p].sum() - p.sum()*(p.sum()+1)/2) / (p.sum()*n.sum())

print("\n[조합 vs 단일 룰]  ← PRD의 핵심 주장 검증")
def ev_rule(mask, tag):
    s = ev[mask]; tp=int(s.is_ato.sum()); al=len(s)
    print(f"  {tag:26s} 경보 {al:>3} | TP {tp:>2} | P {tp/max(al,1):.3f} | R {tp/25:.3f}")
ev_rule(ev.s_new_device==1, "단일: 신규기기")
ev_rule((ev.s_new_device==1)&(ev.s_foreign==1), "단일: 신규기기+해외")
for th in [60,70]: ev_rule(ev.takeover_score>=th, f"가중합 score>={th}")
print(f"\n  단일 신호 최고 AUC {max(roc_auc(ev[c],ev.is_ato) for c,_ in SIGNALS):.3f}"
      f"  vs  가중합 AUC {roc_auc(ev.takeover_score, ev.is_ato):.3f}")

---## 4. 기능③ 환금성 결제 위험 판정 (52회차 + 47회차)52회차 Part2의 IsolationForest 파이프라인을 그대로 쓰고,라벨이 있으므로 47회차의 지도학습(+SHAP)까지 얹는다.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

# ── 피처 (52회차 Part2 §3 그대로 + 환금성·금액이탈 보강)
CASHLIKE = ["해외결제","송금","투자","구독"]      # 환금성·비배송 성격 카테고리
F = pd.DataFrame(index=TX.index)
F["log_amount"]    = np.log1p(TX.amount)
F["hour_sin"]      = np.sin(2*np.pi*TX.hour/24)
F["hour_cos"]      = np.cos(2*np.pi*TX.hour/24)
F["is_weekend"]    = (TX.dayofweek >= 5).astype(int)
F["is_new_device"] = TX.is_new_device
F["is_foreign_ip"] = TX.is_foreign_ip
F["retry_count"]   = TX.retry_count
F["merchant_risk"] = TX.merchant_risk
F["is_cashlike"]   = TX.merchant_category.isin(CASHLIKE).astype(int)
F["amount_ratio"]  = TX.amount_ratio.fillna(1.0)

# 누수 방지 (52회차 co17)
LEAK = {"is_fraud","risk_score","status"}
assert not (set(F.columns) & LEAK), "금지 컬럼 포함!"

pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                 ("scaler",  RobustScaler()),
                 ("model",   IsolationForest(n_estimators=300, contamination=0.02,
                                             random_state=42, n_jobs=-1))])
pipe.fit(F)
TX["anomaly_score"] = -pipe.decision_function(F)      # 부호 뒤집어 '클수록 위험'
TX["is_anomaly"]    = (pipe.predict(F) == -1).astype(int)
print(f"[IsolationForest] 이상 후보 {int(TX.is_anomaly.sum())}건")

# ── 47회차: 지도학습 + SHAP
from sklearn.ensemble import HistGradientBoostingClassifier
order = TX.transaction_time.sort_values().index          # ★ 시간순 분할 (랜덤 금지)
cut = int(len(order)*0.7); tr, te = order[:cut], order[cut:]
clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08,
                                     random_state=42).fit(F.loc[tr], TX.is_fraud[tr])
proba_te = clf.predict_proba(F.loc[te])[:,1]
auc_te = roc_auc(proba_te, TX.is_fraud[te])
print(f"[지도학습] 시간순 분할 ROC-AUC {auc_te:.3f}  (학습 {len(tr)} / 평가 {len(te)})")
print("  ⚠️ 값이 높은 것은 모델이 좋아서가 아니라 **수업 실습 데이터가 쉽기 때문**이다.")
print(f"     사기 12건이 해외결제(사기율 {TX[TX.merchant_category=='해외결제'].is_fraud.mean():.1%})와"
      f" 해외IP(사기율 {TX[TX.is_foreign_ip==1].is_fraud.mean():.1%})에 몰려 있다.")
print(f"     실데이터(IEEE-CIS 59만건) 기준 같은 방식의 ROC-AUC는 0.798이었다. 발표에는 이 대비를 함께 쓴다.")

# ⚠️ 화면·검토큐에 뿌릴 점수는 **out-of-fold**로 뽑는다.
#    clf.predict_proba(F)를 그대로 쓰면 학습에 사용된 행은 모델이 이미 외운 상태라
#    확률이 1.0으로 포화된다(실제로 데모 3건 모두 1.0이 나왔다).
#    "판정 시점에 알 수 없는 정보를 쓰지 않는다"는 이 프로젝트의 원칙은
#    성능 측정뿐 아니라 **화면에 보여주는 점수**에도 똑같이 적용되어야 한다.
from sklearn.model_selection import cross_val_predict, StratifiedKFold
oof = cross_val_predict(
    HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, random_state=42),
    F, TX.is_fraud, cv=StratifiedKFold(5, shuffle=True, random_state=42),
    method="predict_proba")[:, 1]
# 그런데 OOF 확률은 그대로 쓸 수 없다. 사기가 12건뿐이라 **확률 보정이 안 된다**
#   (사기 12건의 OOF 확률 중앙값이 0.000이다).
#   반면 **순위는 유효하다**(OOF 기준 ROC-AUC 0.977, 사기건이 상위 1.5%에 몰림).
# → 화면 점수는 확률이 아니라 **백분위(percentile rank)**를 쓴다.
TX["payment_risk_raw"] = oof
TX["payment_risk"] = pd.Series(oof).rank(pct=True).values

_f = TX.is_fraud == 1
print(f"[표시용 점수] out-of-fold — 확률 최댓값 {oof.max():.3f} / 사기건 확률 중앙값 {np.median(oof[_f]):.3f}")
print(f"  → 확률은 보정 불가(사기 12건). 순위는 유효: OOF ROC-AUC {roc_auc(oof, TX.is_fraud):.3f}")
print(f"  → 백분위로 변환해 사용. 사기건 백분위 중앙값 {TX.loc[_f,'payment_risk'].median():.3f}")
print("  (in-sample 예측을 쓰면 전부 1.0으로 포화되어 화면 점수가 부풀려진다)")

In [ ]:
# SHAP — 47회차 A-7 패턴 (없으면 건너뛴다)
SHAP_OK = False
try:
    import shap
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(F)
    SHAP_OK = True
    print("SHAP 계산 완료:", np.shape(shap_values))
except Exception as e:
    print("SHAP 미설치 → 대체 기여도(피처×상관) 사용. Colab에서는 !pip install shap")
    corr = {c: np.corrcoef(F[c], TX.is_fraud)[0,1] for c in F.columns}
    shap_values = np.array([[corr[c]*F[c].iloc[i] for c in F.columns] for i in range(len(F))])
SHAP_COLS = F.columns.tolist()

---## 5. 통합 점수 — 세 층을 하나로기능①은 **점수를 더하지 않고 임계값을 낮춘다**(PRD 원칙: 노출 정황 단독으로 조치 유발 금지).

In [ ]:
# 결제 건마다 그 시점의 계정 상태를 붙인다
TX = TX.merge(exp[["user_id","exposure_score","exposure_action"]], on="user_id", how="left")

# 그 결제 이전의 최근 takeover_score (누수 방지: 미래 이벤트는 안 봄)
tk = ev[["user_id","change_time","takeover_score"]].sort_values("change_time")
TX = pd.merge_asof(TX.sort_values("transaction_time"), tk.rename(columns={"change_time":"transaction_time"}),
                   on="transaction_time", by="user_id", direction="backward")
TX["takeover_score"] = TX.takeover_score.fillna(0)

# 통합 위험 점수 (0~100)
TX["total_risk"] = (0.6*TX.takeover_score + 40*TX.payment_risk).clip(0,100)

# 기능①은 임계값을 낮추는 방식으로 반영
TX["threshold"] = 60 * (1 - 0.5*TX.exposure_score/100)

def final_action(r):
    if r.total_risk >= r.threshold + 20: return "차단 + 계정 잠금"
    if r.total_risk >= r.threshold:      return "추가 인증"
    if r.total_risk >= r.threshold - 15: return "운영 검토 큐"
    return "정상 처리"
TX["action"] = TX.apply(final_action, axis=1)

print(TX.action.value_counts().to_string())
print(f"\n확정 사기 {int(TX.is_fraud.sum())}건 중 조치된 건: "
      f"{int(TX[(TX.is_fraud==1)&(TX.action!='정상 처리')].shape[0])}건")

### `reason` 생성 — 고객 문구 (판정 근거를 구체적으로 노출하지 않는다)

In [ ]:
PHRASE = {
    "s_multi_ip":    "여러 계정에 접속을 시도한 곳에서 로그인 요청이 있었어요",
    "s_foreign":     "평소 접속하지 않던 지역에서 로그인되었어요",
    "s_contact_chg": "로그인 직후 연락처가 변경되었어요",
    "s_new_device":  "평소 쓰지 않던 기기에서 접속했어요",
    "s_fail_run":    "로그인 실패가 반복된 뒤 성공했어요",
    "E1_own_breach": "계열 서비스에서 회원님 정보가 확인되었어요",
    "E2_darkweb":    "외부에 유출된 정보와 일치하는 항목이 있어요",
    "E3_stuffing_ip":"여러 계정을 시도한 접속처에서 로그인 시도가 있었어요",
    "is_cashlike":   "바로 사용할 수 있는 상품이었어요",
    "log_amount":    "평소보다 큰 금액이었어요",
    "amount_ratio":  "평소 결제 금액과 차이가 컸어요",
    "is_new_device": "평소 쓰지 않던 기기에서 결제했어요",
}

def build_reason(user_id, tx_idx=None, top=4):
    """고객 화면용 근거 목록 [(문구, 기여도)] — 기여도 내림차순 상위 N개"""
    items = []
    e = exp[exp.user_id == user_id]
    if len(e):
        e = e.iloc[0]
        for c, w in [("E1_own_breach",60), ("E2_darkweb",40), ("E3_stuffing_ip",15)]:
            if e[c] and e.gate: items.append((PHRASE[c], w))
    v = ev[ev.user_id == user_id]
    if len(v):
        v = v.iloc[-1]
        for c,_ in SIGNALS:
            if v[c]: items.append((PHRASE[c], WEIGHTS[c]))
    if tx_idx is not None:
        # ⚠️ SHAP 값은 룰 배점(0~100)과 단위가 다르다. 같은 막대에 그리려면 스케일을 맞춰야 한다.
        #    양수 기여의 합이 최대 40점이 되도록 정규화한다(결제 층의 통합 점수 비중과 동일).
        contrib = pd.Series(shap_values[tx_idx], index=SHAP_COLS)
        pos = contrib[contrib > 0]
        if len(pos):
            scaled = pos / pos.sum() * 40
            for c, val in scaled.sort_values(ascending=False).head(2).items():
                if c in PHRASE: items.append((PHRASE[c], round(float(val), 1)))
    return sorted(items, key=lambda x: -x[1])[:top]

demo = ev.sort_values("takeover_score", ascending=False).user_id.iloc[0]
print(f"[예시] {demo}")
for p, s in build_reason(demo): print(f"  {s:>5} · {p}")

---## 6. 운영 검토 큐 CSV — 42회차 대시보드 규격42회차 `app.py`가 요구하는 컬럼은 **`user_id`, `score`, `action`, `reason`** 4개다.이 규격으로 저장하면 대시보드 코드를 고치지 않고 그대로 붙는다.

In [ ]:
OUT = Path("/content/outputs") if Path("/content").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

queue = (TX[TX.action != "정상 처리"]
         .sort_values("total_risk", ascending=False)
         .groupby("user_id").head(1))
queue = queue.assign(
    score  = queue.total_risk.round(1),
    reason = queue.apply(lambda r: ", ".join(p for p,_ in build_reason(r.user_id, r.name)) or "복합 패턴", axis=1)
)[["user_id","score","action","reason"]]

path = OUT/"fds_review_queue.csv"
queue.to_csv(path, index=False, encoding="utf-8-sig")
print(f"저장: {path} / {len(queue)}행  → 42회차 대시보드에 그대로 업로드")
print(queue.head(5).to_string(index=False))

---## 7. 팀원 화면용 데이터 생성`shap-starbucks.html`·`linked-accounts.html`이 쓰는 형태의 JSON을 만든다.화면의 하드코딩 값을 이 파이프라인 산출물로 교체하면 **화면과 로직이 실제로 연결**된다.

In [ ]:
def screen_payload(user_id):
    """팀원 화면(shap-*.html)에 넣을 JSON"""
    v = ev[ev.user_id == user_id]
    e = exp[exp.user_id == user_id].iloc[0]
    t = TX[(TX.user_id == user_id)].sort_values("total_risk", ascending=False)
    top_tx = t.iloc[0] if len(t) else None
    reasons = build_reason(user_id, top_tx.name if top_tx is not None else None, top=5)
    # 종합 점수의 구성을 그대로 담는다 — 화면에서 "76이 어디서 왔는지" 설명할 수 있어야 한다
    tk = float(v.takeover_score.iloc[-1]) if len(v) else 0.0
    pr = float(top_tx.payment_risk) if top_tx is not None else 0.0
    return {
        "user_id": user_id,
        "total_score": int(round(top_tx.total_risk)) if top_tx is not None else 0,
        "score_breakdown": {
            "takeover_part": round(0.6 * tk, 1),      # 0.6 × 계정 장악 점수
            "payment_part":  round(40 * pr, 1),       # 40 × 결제 위험 확률
            "formula": "종합 = 0.6×계정장악 + 40×결제위험확률 (노출 정황은 임계값을 낮추는 데 쓰임)",
        },
        "payment_risk": round(pr, 3),
        "level": ("매우 위험" if top_tx is not None and top_tx.total_risk >= 80 else
                  "위험" if top_tx is not None and top_tx.total_risk >= 60 else "주의"),
        "exposure_score": int(e.exposure_score),
        "takeover_score": int(v.takeover_score.iloc[-1]) if len(v) else 0,
        "target": None if top_tx is None else {
            "time":     top_tx.transaction_time.strftime("%H:%M"),
            "merchant": top_tx.merchant_category,
            "amount":   int(top_tx.amount),
            "channel":  top_tx.channel,
        },
        "reasons": [{"text": p, "contrib": s} for p, s in reasons],
        "action": None if top_tx is None else top_tx.action,
    }

# 데모 케이스 선정
#   단순히 total_risk 상위만 뽑으면 세 건 모두 exposure_score=0이 나와
#   기능①(노출 정황 스코어링)이 화면에 전혀 드러나지 않는다.
#   세 기능이 모두 보이도록 **노출 점수가 있는 건을 최소 1건 포함**한다.
_top = TX.sort_values("total_risk", ascending=False).drop_duplicates("user_id")
_with_exp = _top[_top.exposure_score > 0].head(1).user_id.tolist()
cases = _with_exp + [u for u in _top.user_id.tolist() if u not in _with_exp][:3 - len(_with_exp)]
print("데모 케이스:", cases, "| 노출 점수 있는 건 포함:", len(_with_exp))
payloads = {u: screen_payload(u) for u in cases}
p = OUT/"screen_payload.json"
p.write_text(json.dumps(payloads, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", p, "\n")
print(json.dumps(payloads[cases[0]], ensure_ascii=False, indent=2))

---## 8. 평가 — 불균형 데이터이므로 Accuracy는 쓰지 않는다

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix
y = TX.is_fraud.astype(int); pred = (TX.action != "정상 처리").astype(int)
tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
print(f"경보 {int(pred.sum())}건 | TP {tp} / FP {fp} / FN {fn} / TN {tn}")
print(f"Precision {precision_score(y,pred,zero_division=0):.3f} | Recall {recall_score(y,pred,zero_division=0):.3f}")
print("※ 불균형(사기 0.5%)이라 Accuracy는 무의미 — 전부 정상이라 찍어도 99.5%\n")

scored = TX.sort_values("total_risk", ascending=False).reset_index(drop=True)
tot = int(scored.is_fraud.sum())
print("[Precision@K / Recall@K] — 운영 검토 용량별")
for k in [20, 50, 100, 200]:
    c = int(scored.head(k).is_fraud.sum())
    print(f"  K={k:>3}  사기 {c:>2}건  P@K {c/k:.3f}  R@K {c/tot:.3f}")
print(f"\n기저 사기율 {y.mean():.3%} → K=50에서 리프트 {(int(scored.head(50).is_fraud.sum())/50)/y.mean():.1f}배")

---## 9. 정리 — 발표용이 노트북이 보여주는 것:1. **수업 실습이 제품이 됐다** — 41회차 스코어링 → 기능①·②, 52회차 이상탐지 → 기능③, 47회차 SHAP → 설명, 42회차 대시보드 → 운영 검토 큐2. **누수를 알고 막았다** — `ip_user_count`를 시점 누적으로, 결제 모델을 시간순 분할로3. **조합이 단일 룰을 이긴다** — 같은 경보량에서 정밀도·재현율이 함께 오른다4. **화면과 로직이 연결된다** — `screen_payload.json`이 팀원 화면의 하드코딩 값을 대체한다**한계 (먼저 밝힐 것)**- 로그인·계정변경은 **합성 데이터**다 (`DATA_CARD.md`)- 기능① E1·E2는 **시뮬레이션**이다. E3만 실제 계산- 기능② 배점을 **같은 데이터에서 뽑아 같은 데이터로 평가**했다 → 낙관적